<a href="https://colab.research.google.com/github/anangshachatterjee/anangsha-chatterjee-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will rank content pages for refresh review using two decision-time signals:

1. **Staleness:** pages that have gone a long time since their last update receive a higher priority.
2. **CTR opportunity:** pages with meaningful search visibility, a relatively strong search position, but low CTR receive a higher priority for CTR-focused review.

The rule is intentionally simple and transparent. It is a baseline for comparison with the ML model in Week 5, not a claim that these signals cause traffic decline.

### Reason codes

- `STALE_VISIBLE_PAGE` — the page has been unupdated for a long time and has meaningful impressions.
- `LOW_CTR_VISIBLE_PAGE` — the page has meaningful impressions, is positioned within the first 20 results, but has low CTR.
- `STALE_AND_LOW_CTR` — both conditions are present.
- `GENERAL_REVIEW` — neither specific condition is met.

### Action labels

- `refresh` — prioritize a content refresh review.
- `refresh_and_review_ctr` — prioritize a refresh with special attention to search-result CTR.
- `monitor` — lower-priority review rather than immediate action.

The score uses only information available at the decision moment. The decline label and product-generated flags are used only for signal checking/evaluation, not as score inputs.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Public starter dataset
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
# =========================================================
# CREATE THE EVALUATION LABEL
# =========================================================

# The raw starter data may not contain is_declining_label.
# FlyRank defines the label from trend_direction.

if "is_declining_label" not in df.columns:

    if "trend_direction" not in df.columns:
        raise KeyError(
            "The dataset contains neither 'is_declining_label' "
            "nor 'trend_direction'. Check the loaded dataset."
        )

    df["is_declining_label"] = (
        df["trend_direction"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "down"
    )

print("✅ is_declining_label is ready")

print(
    df["is_declining_label"]
    .value_counts()
)

✅ is_declining_label is ready
is_declining_label
True     16262
False    13738
Name: count, dtype: int64


In [5]:
# =========================================================
# SIGNAL CHECKS
# =========================================================

# is_declining_label is used ONLY for evaluating signals.
# It is NOT used in the baseline score.


# ---------------------------------------------------------
# SIGNAL 1 — STALENESS
# ---------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, np.inf],
    labels=[
        "<90 days",
        "90-179 days",
        "180+ days"
    ]
)

staleness_check = (
    df.groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        declining_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

staleness_check["declining_rate_pct"] = (
    staleness_check["declining_rate"] * 100
).round(2)

print("SIGNAL 1 — STALENESS")
display(staleness_check)


# ---------------------------------------------------------
# SIGNAL 2 — CTR VS POSITION
# ---------------------------------------------------------

df["ctr_position_bucket"] = np.select(
    [
        (
            (df["avg_position"] > 0) &
            (df["avg_position"] <= 20) &
            (df["ctr"] < 0.5)
        ),

        (
            (df["avg_position"] > 0) &
            (df["avg_position"] <= 20) &
            (df["ctr"] >= 0.5)
        ),

        (
            df["avg_position"] > 20
        )
    ],
    [
        "Top-20 + low CTR",
        "Top-20 + higher CTR",
        "Position >20"
    ],
    default="No usable position"
)

ctr_position_check = (
    df.groupby(
        "ctr_position_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        declining_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

ctr_position_check["declining_rate_pct"] = (
    ctr_position_check["declining_rate"] * 100
).round(2)

print("\nSIGNAL 2 — CTR VS POSITION")
display(ctr_position_check)

SIGNAL 1 — STALENESS


,staleness_bucket,n,declining_rate,declining_rate_pct
0,<90 days,20655,0.512031,51.20
1,90-179 days,9171,0.611057,61.11
2,180+ days,174,0.471264,47.13



SIGNAL 2 — CTR VS POSITION


,ctr_position_bucket,n,declining_rate,declining_rate_pct
0,No usable position,1205,0.006639,0.66
1,Position >20,8539,0.528165,52.82
2,Top-20 + higher CTR,3698,0.487561,48.76
3,Top-20 + low CTR,16558,0.600374,60.04


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# =========================================================
# 2. BUILD THE RANKED QUEUE
# =========================================================
#
# One simple, transparent baseline rule.
#
# Inputs used:
#   1. days_since_last_update
#   2. ctr
#   3. avg_position
#   4. impressions_90d
#
# IMPORTANT:
# is_declining_label is NOT used here.
# trend_direction is NOT used here.
# trend_pct is NOT used here.
# FlyRank product flags are NOT used here.
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Check that the decision-time fields exist
# ---------------------------------------------------------

required_features = [
    "content_id",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

missing_features = [
    col for col in required_features
    if col not in df.columns
]

if missing_features:
    raise KeyError(
        f"Missing required columns: {missing_features}"
    )

print("✅ Required baseline columns are available.")


# ---------------------------------------------------------
# 2. Create a separate baseline dataframe
# ---------------------------------------------------------

baseline = df.copy()


# ---------------------------------------------------------
# 3. Staleness component
# ---------------------------------------------------------
#
# Cap staleness at 365 days so extremely old pages do not
# dominate the score indefinitely.
#
# 0 = recently updated
# 1 = 365+ days since update
# ---------------------------------------------------------

baseline["staleness_component"] = (
    baseline["days_since_last_update"]
    .clip(lower=0, upper=365)
    / 365
)


# ---------------------------------------------------------
# 4. CTR opportunity component
# ---------------------------------------------------------
#
# We give priority to pages that:
#   - have useful visibility
#   - rank within the Top 20
#   - have relatively low CTR
#
# CTR threshold = 0.5
#
# The score increases as CTR falls below 0.5.
# ---------------------------------------------------------

baseline["ctr_opportunity_component"] = np.where(

    (
        (baseline["impressions_90d"] >= 100) &
        (baseline["avg_position"] > 0) &
        (baseline["avg_position"] <= 20)
    ),

    (
        (0.5 - baseline["ctr"])
        .clip(lower=0)
        / 0.5
    ),

    0.0
)


# ---------------------------------------------------------
# 5. Final baseline score
# ---------------------------------------------------------
#
# 60% staleness
# 40% CTR opportunity
#
# Score is between 0 and 1.
# Higher score = higher review priority.
# ---------------------------------------------------------

baseline["baseline_score"] = (
    0.60 * baseline["staleness_component"]
    +
    0.40 * baseline["ctr_opportunity_component"]
)


# ---------------------------------------------------------
# 6. Assign ONE reason code
# ---------------------------------------------------------

baseline["reason_code"] = np.select(

    [
        (
            (baseline["days_since_last_update"] >= 180) &
            (baseline["impressions_90d"] >= 100) &
            (baseline["avg_position"] > 0) &
            (baseline["avg_position"] <= 20) &
            (baseline["ctr"] < 0.5)
        ),

        (
            (baseline["days_since_last_update"] >= 180) &
            (baseline["impressions_90d"] >= 100)
        ),

        (
            (baseline["impressions_90d"] >= 100) &
            (baseline["avg_position"] > 0) &
            (baseline["avg_position"] <= 20) &
            (baseline["ctr"] < 0.5)
        )
    ],

    [
        "STALE_AND_LOW_CTR",
        "STALE_VISIBLE_PAGE",
        "LOW_CTR_VISIBLE_PAGE"
    ],

    default="GENERAL_REVIEW"
)


# ---------------------------------------------------------
# 7. Assign ONE action label
# ---------------------------------------------------------

baseline["action"] = np.select(

    [
        baseline["reason_code"] == "STALE_AND_LOW_CTR",

        baseline["reason_code"] == "LOW_CTR_VISIBLE_PAGE",

        baseline["reason_code"] == "STALE_VISIBLE_PAGE"
    ],

    [
        "refresh_and_review_ctr",
        "refresh_and_review_ctr",
        "refresh"
    ],

    default="monitor"
)


# ---------------------------------------------------------
# 8. Rank the complete queue
# ---------------------------------------------------------

baseline = baseline.sort_values(
    by=["baseline_score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

baseline["baseline_rank"] = (
    np.arange(len(baseline)) + 1
)


# ---------------------------------------------------------
# 9. Create the final queue
# ---------------------------------------------------------

queue_columns = [
    "baseline_rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

# Add client_id only if it exists
if "client_id" in baseline.columns:
    queue_columns.insert(2, "client_id")

queue = baseline[queue_columns].copy()


# ---------------------------------------------------------
# 10. Write the required CSV
# ---------------------------------------------------------

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# 11. Display results
# ---------------------------------------------------------

print("\n✅ Baseline queue created successfully.")
print("Rows ranked:", len(queue))
print("Output file:", output_path)

print("\nTop 10 ranked pages:")
display(queue.head(10))

✅ Required baseline columns are available.

✅ Baseline queue created successfully.
Rows ranked: 30000
Output file: work/outputs/baseline_action_score.csv

Top 10 ranked pages:


,baseline_rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position
0,1,content_02b0d6e30129,client_19581e27de,0.914521,STALE_AND_LOW_CTR,refresh_and_review_ctr,313,176,0.00,6.9
1,2,content_f488400fca67,client_9400f1b21c,0.901370,STALE_AND_LOW_CTR,refresh_and_review_ctr,305,155,0.00,5.7
2,3,content_ab27c30d81f4,client_4ec9599fc2,0.899726,STALE_AND_LOW_CTR,refresh_and_review_ctr,304,103,0.00,8.9
3,4,content_4f241bad48a3,client_6208ef0f77,0.787945,STALE_AND_LOW_CTR,refresh_and_review_ctr,236,285,0.00,19.1
4,5,content_72496874f806,client_4ec9599fc2,0.702795,STALE_AND_LOW_CTR,refresh_and_review_ctr,301,821,0.24,5.8
5,6,content_958a46db26bd,client_d029fa3a95,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,198,0.00,6.7
6,7,content_b6e4581523ed,client_d029fa3a95,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,104,0.00,14.0
7,8,content_ea41fe5cf292,client_d029fa3a95,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,265,0.00,8.0
8,9,content_fd16e3475c29,client_d029fa3a95,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,429,0.00,9.0
9,10,content_928af3e22c80,client_7f2253d7e2,0.621260,STALE_AND_LOW_CTR,refresh_and_review_ctr,193,1697,0.12,15.8


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# =========================================================
# 3. TOP-20 REVIEW
# =========================================================
#
# For every top-20 recommendation we record:
#   - action
#   - reason code
#   - confidence note
#   - what would make the recommendation wrong
#
# This is a human-review step.
# The baseline is decision-support, not proof that an action
# is definitely correct.
# =========================================================


# ---------------------------------------------------------
# 1. Select the top 20
# ---------------------------------------------------------

top20 = baseline.head(20).copy()

print("Top 20 baseline recommendations:")
display(top20[
    [
        "baseline_rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
])


# ---------------------------------------------------------
# 2. Confidence note
# ---------------------------------------------------------

def confidence_note(row):

    score = row["baseline_score"]

    if score >= 0.75:
        return (
            "Higher confidence: the page has a relatively strong "
            "combination of the signals used by this baseline."
        )

    elif score >= 0.50:
        return (
            "Moderate confidence: the signals support review, "
            "but they are not decisive on their own."
        )

    else:
        return (
            "Lower confidence: this page is near the weaker end "
            "of the ranked queue and should be reviewed cautiously."
        )


# ---------------------------------------------------------
# 3. What would make the recommendation wrong?
# ---------------------------------------------------------

def wrong_if(row):

    reasons = []

    # Staleness can be intentional.
    if row["days_since_last_update"] >= 180:
        reasons.append(
            "the content was intentionally left unchanged "
            "and is still appropriate despite its age"
        )

    # Low CTR may have explanations not represented by this
    # simple baseline.
    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append(
            "the observed CTR is appropriate for the query/page "
            "context even though it is below this baseline threshold"
        )

    # Low search exposure makes the recommendation less useful.
    if row["impressions_90d"] < 100:
        reasons.append(
            "the page has too little search exposure for a strong "
            "CTR-based recommendation"
        )

    # If none of the specific conditions apply.
    if not reasons:
        reasons.append(
            "the observed signals are temporary or additional "
            "context would show that the recommended action is unnecessary"
        )

    return "; ".join(reasons)


# ---------------------------------------------------------
# 4. Create the review table
# ---------------------------------------------------------

review = top20[
    [
        "baseline_rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
].copy()


review["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

review["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)


# ---------------------------------------------------------
# 5. Display the completed Top-20 review
# ---------------------------------------------------------

print("\nTOP-20 HUMAN REVIEW")
display(review)

Top 20 baseline recommendations:


,baseline_rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position
0,1,content_02b0d6e30129,0.914521,STALE_AND_LOW_CTR,refresh_and_review_ctr,313,176,0.00,6.9
1,2,content_f488400fca67,0.901370,STALE_AND_LOW_CTR,refresh_and_review_ctr,305,155,0.00,5.7
2,3,content_ab27c30d81f4,0.899726,STALE_AND_LOW_CTR,refresh_and_review_ctr,304,103,0.00,8.9
3,4,content_4f241bad48a3,0.787945,STALE_AND_LOW_CTR,refresh_and_review_ctr,236,285,0.00,19.1
4,5,content_72496874f806,0.702795,STALE_AND_LOW_CTR,refresh_and_review_ctr,301,821,0.24,5.8
5,6,content_958a46db26bd,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,198,0.00,6.7
6,7,content_b6e4581523ed,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,104,0.00,14.0
7,8,content_ea41fe5cf292,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,265,0.00,8.0
8,9,content_fd16e3475c29,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,429,0.00,9.0
9,10,content_928af3e22c80,0.621260,STALE_AND_LOW_CTR,refresh_and_review_ctr,193,1697,0.12,15.8



TOP-20 HUMAN REVIEW


,baseline_rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position,confidence_note,what_would_make_it_wrong
0,1,content_02b0d6e30129,0.914521,STALE_AND_LOW_CTR,refresh_and_review_ctr,313,176,0.00,6.9,Higher confidence: the page has a relatively s...,the content was intentionally left unchanged a...
1,2,content_f488400fca67,0.901370,STALE_AND_LOW_CTR,refresh_and_review_ctr,305,155,0.00,5.7,Higher confidence: the page has a relatively s...,the content was intentionally left unchanged a...
2,3,content_ab27c30d81f4,0.899726,STALE_AND_LOW_CTR,refresh_and_review_ctr,304,103,0.00,8.9,Higher confidence: the page has a relatively s...,the content was intentionally left unchanged a...
3,4,content_4f241bad48a3,0.787945,STALE_AND_LOW_CTR,refresh_and_review_ctr,236,285,0.00,19.1,Higher confidence: the page has a relatively s...,the content was intentionally left unchanged a...
4,5,content_72496874f806,0.702795,STALE_AND_LOW_CTR,refresh_and_review_ctr,301,821,0.24,5.8,Moderate confidence: the signals support revie...,the content was intentionally left unchanged a...
5,6,content_958a46db26bd,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,198,0.00,6.7,Moderate confidence: the signals support revie...,the content was intentionally left unchanged a...
6,7,content_b6e4581523ed,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,104,0.00,14.0,Moderate confidence: the signals support revie...,the content was intentionally left unchanged a...
7,8,content_ea41fe5cf292,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,265,0.00,8.0,Moderate confidence: the signals support revie...,the content was intentionally left unchanged a...
8,9,content_fd16e3475c29,0.700822,STALE_AND_LOW_CTR,refresh_and_review_ctr,183,429,0.00,9.0,Moderate confidence: the signals support revie...,the content was intentionally left unchanged a...
9,10,content_928af3e22c80,0.621260,STALE_AND_LOW_CTR,refresh_and_review_ctr,193,1697,0.12,15.8,Moderate confidence: the signals support revie...,the content was intentionally left unchanged a...


In [8]:
# =========================================================
# READABLE TOP-20 REVIEW
# =========================================================

for _, row in review.iterrows():

    print(
        f"#{int(row['baseline_rank'])} "
        f"{row['content_id']} | "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']}"
    )

    print(
        f"  Confidence: {row['confidence_note']}"
    )

    print(
        f"  What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

    print("-" * 80)

#1 content_02b0d6e30129 | Action: refresh_and_review_ctr | Reason: STALE_AND_LOW_CTR
  Confidence: Higher confidence: the page has a relatively strong combination of the signals used by this baseline.
  What would make it wrong: the content was intentionally left unchanged and is still appropriate despite its age; the observed CTR is appropriate for the query/page context even though it is below this baseline threshold
--------------------------------------------------------------------------------
#2 content_f488400fca67 | Action: refresh_and_review_ctr | Reason: STALE_AND_LOW_CTR
  Confidence: Higher confidence: the page has a relatively strong combination of the signals used by this baseline.
  What would make it wrong: the content was intentionally left unchanged and is still appropriate despite its age; the observed CTR is appropriate for the query/page context even though it is below this baseline threshold
-------------------------------------------------------------------------

In [9]:
print("Number of rows reviewed:", len(review))

assert len(review) == 20, "Top-20 review should contain exactly 20 rows."

required_review_columns = [
    "baseline_rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]

missing = [
    col for col in required_review_columns
    if col not in review.columns
]

assert not missing, f"Missing review columns: {missing}"

print("✅ Top-20 review is complete.")

Number of rows reviewed: 20
✅ Top-20 review is complete.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A useful baseline should not be treated as perfect. I review highly ranked rows that appear questionable because their score may be driven by only one strong signal or because they have limited search exposure.

I also check the baseline inputs against known outcome/product fields. The baseline must not use future-window outcomes, existing FlyRank flags, or label-derived fields because those would make the ranking artificially strong and unavailable at decision time.

In [11]:
# =========================================================
# 4. WEAK-PICK REVIEW
# =========================================================

# A weak pick is a highly ranked page where the baseline
# recommendation is supported by only limited evidence.

score_cutoff = baseline["baseline_score"].quantile(0.90)

weak_picks = baseline[
    (baseline["baseline_score"] >= score_cutoff)
    &
    (
        (baseline["days_since_last_update"] < 180)
        |
        (baseline["impressions_90d"] < 100)
        |
        (
            (baseline["avg_position"] > 20)
        )
    )
].head(10).copy()

print("Potential weak picks:")

display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ]
)

Potential weak picks:


,baseline_rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
10,11,content_1b4ec72dafd4,0.600000,GENERAL_REVIEW,monitor,372,2,7.0,0.0
11,12,content_3f3576c295f5,0.600000,GENERAL_REVIEW,monitor,373,1,1.0,100.0
12,13,content_55a5b1c46474,0.600000,GENERAL_REVIEW,monitor,373,35,7.5,0.0
13,14,content_8d56efff1e71,0.600000,GENERAL_REVIEW,monitor,372,1,35.0,0.0
14,15,content_f6fdf87348f6,0.600000,GENERAL_REVIEW,monitor,373,2,32.5,0.0
16,17,content_0868e82484db,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,106,1149,4.1,0.0
17,18,content_20c82b1f08ac,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,106,256,11.1,0.0
18,19,content_26d48a980581,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,106,1266,4.6,0.0
19,20,content_396217b0dc41,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,106,1032,10.8,0.0
20,21,content_72ee9589636c,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,106,229,17.4,0.0


In [12]:
# =========================================================
# WHY COULD THESE PICKS BE WRONG?
# =========================================================

def weak_pick_reason(row):

    reasons = []

    if row["days_since_last_update"] < 180:
        reasons.append(
            "the page is not especially stale"
        )

    if row["impressions_90d"] < 100:
        reasons.append(
            "the page has limited search exposure"
        )

    if row["avg_position"] > 20:
        reasons.append(
            "the page is outside the Top 20, "
            "so the CTR signal is less actionable"
        )

    if not reasons:
        reasons.append(
            "the simple baseline may not capture "
            "important page-specific context"
        )

    return "; ".join(reasons)


if len(weak_picks) > 0:

    weak_picks["why_it_may_be_wrong"] = (
        weak_picks.apply(
            weak_pick_reason,
            axis=1
        )
    )

    display(
        weak_picks[
            [
                "baseline_rank",
                "content_id",
                "baseline_score",
                "reason_code",
                "action",
                "why_it_may_be_wrong"
            ]
        ]
    )

else:
    print(
        "No high-scoring weak picks met the "
        "review criteria. This result is retained honestly."
    )

,baseline_rank,content_id,baseline_score,reason_code,action,why_it_may_be_wrong
10,11,content_1b4ec72dafd4,0.600000,GENERAL_REVIEW,monitor,the page has limited search exposure
11,12,content_3f3576c295f5,0.600000,GENERAL_REVIEW,monitor,the page has limited search exposure
12,13,content_55a5b1c46474,0.600000,GENERAL_REVIEW,monitor,the page has limited search exposure
13,14,content_8d56efff1e71,0.600000,GENERAL_REVIEW,monitor,the page has limited search exposure; the page...
14,15,content_f6fdf87348f6,0.600000,GENERAL_REVIEW,monitor,the page has limited search exposure; the page...
16,17,content_0868e82484db,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,the page is not especially stale
17,18,content_20c82b1f08ac,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,the page is not especially stale
18,19,content_26d48a980581,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,the page is not especially stale
19,20,content_396217b0dc41,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,the page is not especially stale
20,21,content_72ee9589636c,0.574247,LOW_CTR_VISIBLE_PAGE,refresh_and_review_ctr,the page is not especially stale


In [13]:
# =========================================================
# LEAKAGE CHECK
# =========================================================

# Fields that must never be baseline score inputs.
leakage_fields = [
    "is_declining",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "health_score",
    "recommended_action",
    "action_type",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "is_underperformer",
    "is_initial_refresh_candidate"
]


# These are the actual raw inputs used by our Section 2 score.
score_inputs = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]


leaked_inputs = sorted(
    set(score_inputs).intersection(
        leakage_fields
    )
)


print("LEAKAGE CHECK")
print("=" * 50)

print("Score inputs:")
for col in score_inputs:
    print("  -", col)

print("\nLeakage-risk fields:")
for col in leakage_fields:
    print("  -", col)

print("\nFields accidentally used in score:")
print(leaked_inputs)


assert leaked_inputs == [], (
    "Leakage detected: "
    + str(leaked_inputs)
)


print("\n✅ No product flags or outcome-derived fields "
      "are used in the baseline score.")

print("✅ No future-window fields are used in the "
      "baseline score.")

print("✅ Baseline inputs are decision-time-safe.")

LEAKAGE CHECK
Score inputs:
  - days_since_last_update
  - impressions_90d
  - ctr
  - avg_position

Leakage-risk fields:
  - is_declining
  - is_declining_label
  - trend_direction
  - trend_pct
  - health_score
  - recommended_action
  - action_type
  - is_quick_win
  - needs_ctr_fix
  - needs_engagement_fix
  - is_underperformer
  - is_initial_refresh_candidate

Fields accidentally used in score:
[]

✅ No product flags or outcome-derived fields are used in the baseline score.
✅ No future-window fields are used in the baseline score.
✅ Baseline inputs are decision-time-safe.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.